# 配置自己的通信测试

上一节跑通了 2 个 rank 的 ReduceScatter。这一节把这些配置拆开看清楚：
集群里有什么、这次选谁通信、测试进程怎样找到它们。

学完你能把参与者从两个改成四个，也能分清"同一台模拟服务器上的四个 rank"
和"分布在两台模拟服务器上的四个 rank"。

## 本节学习大纲

- 先分清集群、通信域和测试进程
- 读取集群配置并启动环境
- 用通信域配置选择参与者
- 查看生成文件与 rank 映射
- 环境变量从哪来
- 让 MPI 进程数与通信域对应
- 遇到问题先看这里

## 1. 先分清集群、通信域和测试进程

配置一次测试要回答三个问题：

| 配置 | 回答什么问题 | 在哪 |
| :--- | :--- | :--- |
| 集群配置 | 模拟环境中有哪些服务器、设备和连接？ | `config/cluster/`，启动时交给 `hccl-vm start` |
| 通信域配置 | 这一次从集群中选哪些设备共同通信？ | `config/topo_meta/`，交给 `hccl-vm mock-comm` |
| 测试参数 | 启动几个进程、执行哪个算子、多少数据？ | `mpirun` 与测试用例的参数 |

**集群规模是可选范围，通信域才是本次实际参与的范围。**
例如模拟集群里有 32 台服务器，这次仍然可以只选其中一台的两个设备。

| 通信域 | 选了谁 | rank 数 |
| :--- | :--- | :--- |
| `112` | pod 0 / server 0 的本地设备 0、1 | 2 |
| `114` | pod 0 / server 0 的本地设备 0 到 3 | 4 |
| `122` | pod 0 的 server 0、1，各取本地设备 0、1 | 4 |

三个都来自同一个集群，但参与者数量或位置不同。

## 2. 读取集群配置并启动环境

### 2.1 准备环境

云端版的路径变量都集中在 `env.sh` 里，每个单元开头 `source` 一次即可。

In [ ]:
%%bash
source ~/hccl_vm_pack/env.sh

echo "PACK         = $PACK"
echo "VM_INSTALL   = $VM_INSTALL"
echo "CLUSTER_YAML = $CLUSTER_YAML"
mkdir -p "$HCCL_VM_RESULT_DIR"

`HCCL_VM_INSTALL_DIR` 是工具的安装根目录，后面从 `bin/` 切到别的目录时，
仍然可以用它访问配置和数据。`HCCL_VM_RESULT_DIR` 用来存本节的测试记录。

### 2.2 看清集群的结构字段

预置集群 YAML 很长（32 台服务器逐条配置），下面只挑几类关键字段看：
集群名称与规模、超节点编号、服务器编号范围、服务器内部拓扑文件。

In [ ]:
%%bash
source ~/hccl_vm_pack/env.sh
grep -nE "^(name|super_node_num|server_num):|super_pod_id|id_range|server_topo" "$VM_INSTALL/config/cluster/$CLUSTER_YAML.yaml" | head -12

| 字段 | 当前配置 | 怎么理解 |
| :--- | :--- | :--- |
| `super_node_num` | `4` | 集群有 4 个超节点，编号 0 到 3 |
| `server_num` | `32` | 集群共 32 台服务器 |
| `super_pod_id` | 每条服务器配置所属的超节点 | `0` 表示第 0 个超节点 |
| `id_range: [0, 7]` | 每个超节点配置服务器 0 到 7 | 两端都包括，共 8 台；4 × 8 = 32 |
| `server_topo` | `ascend950_server_topo_normal.yaml` | 指定每台服务器内部的设备、端口和连接方式 |

`server_topo` 引用的文件在 `config/server_or_pod/`，描述设备数量、端口用途、端口分组和设备间链路。
本节先用预置连接，只调整参与者。

> 上面只显示了前几条（`head -12`）。完整的服务器列表在 YAML 里逐条列出，
> 用 `grep -c "super_pod_id" 那个文件` 可以数出总共有多少条超节点配置。

### 2.3 启动集群

`start` 是常驻进程，仍然用 tmux 放到后台。

In [ ]:
%%bash
source ~/hccl_vm_pack/env.sh

tmux kill-server 2>/dev/null
# kill-server 是异步的：等旧服务端真正退出，否则 new-session 会撞上"正在死的 server"
for i in $(seq 1 20); do
    tmux list-sessions >/dev/null 2>&1 || break
    sleep 0.2
done
for i in 1 2 3; do
    tmux new-session -d -s hvm -x 200 -y 50 "$PACK/run_hccl_vm.sh start $CLUSTER_YAML 2>&1 | tee /tmp/hvm_start.log" && break
    sleep 1
done
if ! tmux has-session -t hvm 2>/dev/null; then
    echo "!! session not started, start output below:"
    tail -20 /tmp/hvm_start.log 2>/dev/null || true
fi

for i in $(seq 1 40); do
    tmux capture-pane -pt hvm 2>/dev/null | grep -q "listening" && break
    sleep 1
done

tmux capture-pane -pt hvm | grep -E "PLUGIN_TAG|checker|listening" || true

看到 `checker ... RUNNING` 和 `HOST Server listening...` 就说明集群起来了。

> 这里的"模拟服务器"只是进程里的数据结构，不要求你真的准备 32 台电脑。

## 3. 用通信域配置选择参与者

### 3.1 先读 112.yaml

`mock-comm 112` 会读取 `config/topo_meta/112.yaml`。先看这个文件：

In [ ]:
%%bash
source ~/hccl_vm_pack/env.sh
sed -n '/^meta:/,$p' "$VM_INSTALL/config/topo_meta/112.yaml"

`meta` 统计本次选中的超节点、服务器和 rank 总数；`topology` 列出具体位置。
这里选的是超节点 0 的服务器 0，使用本地设备编号 0、1，工具再给它们分配通信域内的 rank 0、1。

**本地编号和 rank 编号负责不同的事**：本地编号标识一台服务器上的设备；
rank 编号标识它在本次通信域中的身份。跨服务器时两台服务器都可以有本地设备 0，
但通信域内的 rank 编号各不相同。

### 3.2 再对比 114 和 122

In [ ]:
%%bash
source ~/hccl_vm_pack/env.sh
echo "=== 114 ==="
sed -n '/^meta:/,$p' "$VM_INSTALL/config/topo_meta/114.yaml"
echo "=== 122 ==="
sed -n '/^meta:/,$p' "$VM_INSTALL/config/topo_meta/122.yaml"

| 文件 | `podNum` | `serNum` | `rankNum` | 实际选择 | 对应 MPI 进程数 |
| --- | ---: | ---: | ---: | --- | ---: |
| `112.yaml` | 1 | 1 | 2 | pod 0 / server 0 的本地设备 0、1 | 2 |
| `114.yaml` | 1 | 1 | 4 | pod 0 / server 0 的本地设备 0 到 3 | 4 |
| `122.yaml` | 1 | 2 | 4 | pod 0 的 server 0、1，各取本地设备 0、1 | 4 |

**选配置时读文件里的 `rankNum` 和 `topology`，不要只凭文件名判断。**

先选 112，此时只准备通信域，还没有运行测试用例。

In [ ]:
%%bash
source ~/hccl_vm_pack/env.sh

tmux send-keys -t hvm "$VM mock-comm 112" Enter
for i in $(seq 1 20); do
    tmux capture-pane -pt hvm | grep -q "config by 112.yaml" && break
    sleep 1
done
tmux capture-pane -pt hvm | tail -5

看到 `Initializing mock communication domain: Config=112`，最后出现 `config by 112.yaml`，
说明通信域切换成功。`data/ranktable.json` 就是这一步生成的。

> 中间的 WARNING（`topo.json` 已存在、将被覆盖）是正常现象——每次切换通信域都会重新复制拓扑文件。

## 4. 查看生成文件与 rank 映射

### 4.1 两个 JSON 各管什么

In [ ]:
%%bash
source ~/hccl_vm_pack/env.sh
ls -l "$VM_INSTALL/data/topo.json" "$VM_INSTALL/data/ranktable.json"

| 文件 | 作用 | 重点字段 |
| :--- | :--- | :--- |
| `data/topo.json` | 描述设备、端口和链路；从启动时建立的服务器拓扑复制而来 | `peer_list`、`edge_list`、链路的 `net_layer`、`link_type` |
| `data/ranktable.json` | 描述本次通信域内各 rank 的设备位置与网络地址 | `rank_count`、`rank_list`，每个 rank 的 `rank_id`、`local_id`、`level_list` |

这两个文件由工具生成。改参与者时改通信域 YAML 再跑 `mock-comm`，不用手工拼 JSON。

下面提取 `topo.json` 的第一条链路，看看它描述的是什么。

In [ ]:
%%bash
source ~/hccl_vm_pack/env.sh
python3 - "$VM_INSTALL/data/topo.json" <<'PY'
import json, pathlib, sys
data = json.loads(pathlib.Path(sys.argv[1]).read_text())
print("top keys:", ", ".join(data))
print("peers:", data.get("peer_count"), "edges:", data.get("edge_count"))
edge = data["edge_list"][0]
keys = ("net_layer", "link_type", "local_a", "local_b", "local_a_ports", "local_b_ports")
print(json.dumps({k: edge[k] for k in keys if k in edge}, indent=2))
PY

`topo.json` 描述的是**一台模拟服务器内部的完整拓扑**（每台模拟服务器都按同一份服务器拓扑生成），
与本次通信域选了几个设备无关；选择范围体现在 `ranktable.json` 里。

第一条链路通常是 `PEER2PEER`，`local_a: 0`、`local_b: 1`，
即服务器内的设备 0 与设备 1 直连，两端各记录所用端口。

为了反复对比不同通信域，下面定义一个 `show_ranks`，把每个 rank 的关键字段打印成一行。

> 注意：每个单元是独立的 shell，**函数不能跨单元保留**，所以下面每次用到都重新定义一遍。

In [ ]:
%%bash
source ~/hccl_vm_pack/env.sh

show_ranks() {
  python3 - "$VM_INSTALL/data/ranktable.json" <<'PY'
import json, pathlib, sys
data = json.loads(pathlib.Path(sys.argv[1]).read_text())
print("rank_count =", data["rank_count"])
for rank in data["rank_list"]:
    layer0 = next(item for item in rank["level_list"] if item["net_layer"] == 0)
    print(f"rank {rank['rank_id']} -> {layer0['net_instance_id']}"
          f" / local_id {rank['local_id']} / device_id {rank['device_id']}")
PY
}
show_ranks

期望 `rank_count = 2`，两个 rank 都属于 `sp_0_srv_0_idx_0`（超节点 0 的服务器 0），
本地编号分别为 0、1——与 `112.yaml` 的选择一致。

### 4.2 换成四个参与者，观察它们的位置

先切到 114，再切到 122，每次都用 `show_ranks` 看 rank 落在哪台服务器。

In [ ]:
%%bash
source ~/hccl_vm_pack/env.sh

show_ranks() {
  python3 - "$VM_INSTALL/data/ranktable.json" <<'PY'
import json, pathlib, sys
data = json.loads(pathlib.Path(sys.argv[1]).read_text())
print("rank_count =", data["rank_count"])
for rank in data["rank_list"]:
    layer0 = next(item for item in rank["level_list"] if item["net_layer"] == 0)
    print(f"rank {rank['rank_id']} -> {layer0['net_instance_id']}"
          f" / local_id {rank['local_id']} / device_id {rank['device_id']}")
PY
}

tmux send-keys -t hvm "$VM mock-comm 114" Enter
for i in $(seq 1 20); do
    tmux capture-pane -pt hvm | grep -q "config by 114.yaml" && break
    sleep 1
done
tmux capture-pane -pt hvm | tail -2
echo "--- rank map (114) ---"
show_ranks

`rank_count = 4`，rank 0 到 3 全部属于 `sp_0_srv_0_idx_0`（服务器 0），本地编号 0 到 3。
这就是「同一台模拟服务器上的四个 rank」。

再切到 122：

In [ ]:
%%bash
source ~/hccl_vm_pack/env.sh

show_ranks() {
  python3 - "$VM_INSTALL/data/ranktable.json" <<'PY'
import json, pathlib, sys
data = json.loads(pathlib.Path(sys.argv[1]).read_text())
print("rank_count =", data["rank_count"])
for rank in data["rank_list"]:
    layer0 = next(item for item in rank["level_list"] if item["net_layer"] == 0)
    print(f"rank {rank['rank_id']} -> {layer0['net_instance_id']}"
          f" / local_id {rank['local_id']} / device_id {rank['device_id']}")
PY
}

tmux send-keys -t hvm "$VM mock-comm 122" Enter
for i in $(seq 1 20); do
    tmux capture-pane -pt hvm | grep -q "config by 122.yaml" && break
    sleep 1
done
tmux capture-pane -pt hvm | tail -2
echo "--- rank map (122) ---"
show_ranks

还是 4 个 rank，但分布变了：rank 0、1 属于 `sp_0_srv_0_idx_0`（服务器 0），
rank 2、3 属于 `sp_0_srv_1_idx_0`（服务器 1）。

服务器 1 的本地编号从 0 重新开始，而通信域内的 rank 编号继续是 2、3。

> 每次 `mock-comm` 都会重置通信域和旧的通信任务记录。已经跑过测试时，
> 先完成 Checker 校验并导出结果，再切到下一个通信域。

## 5. 环境变量从哪来

本地版要手工 `export` 一堆变量；云端版这些都在 `env.sh` 里，由 `run_hccl_vm.sh` 和 `env.sh` 一起提供。

下面把关键变量打出来，确认它们指向包内路径。

In [ ]:
%%bash
source ~/hccl_vm_pack/env.sh
printf 'PACK                 = %s\n' "$PACK"
printf 'HCCL_VM_INSTALL_DIR  = %s\n' "$HCCL_VM_INSTALL_DIR"
printf 'RANK_TABLE_FILE      = %s\n' "${RANK_TABLE_FILE:-(set inside the tmux session)}"
test -f "$VM_INSTALL/data/ranktable.json" && echo "ranktable.json exists"
command -v mpirun
mpirun --version | head -1

| 配置 | 云端要求 | 出错时会怎样 |
| :--- | :--- | :--- |
| `LD_LIBRARY_PATH` | 含包内 `lib64` | 启动时报动态库找不到 |
| `ASCEND_HOME_PATH` | 指向包根 `$PACK` | 找不到 `opp/` 与 `lib64/` 里的文件 |
| `RANK_TABLE_FILE` | 包内 `data/ranktable.json` | 用了旧通信域或找不到配置 |
| `HCCL_OP_EXPANSION_MODE` | `CCU_SCHED` | 执行模式与预期不同 |
| `mpirun` | OpenMPI | 启动参数与 MPI 实现不匹配 |

`RANK_TABLE_FILE` 用完整路径，切换工作目录不会把它变成 `bin/data/ranktable.json`。
112、114、122 更新的是同一个文件，路径不变，内容随通信域变化。

## 6. 让 MPI 进程数与通信域对应

### 6.1 先确定参与者数量，再运行测试用例

当前最后选的是 122，`rank_count` 为 4，因此用 `mpirun -np 4` 启动四个进程。
它们都跑在当前实例上，HCCL-VM 把它们映射到前面选好的两台模拟服务器。

| 想改什么 | 改哪里 | 还要一起改什么 |
| :--- | :--- | :--- |
| 两个 rank 改四个 | `mock-comm 112` → `114` 或 `122` | `mpirun -np 2` → `-np 4` |
| 四个 rank 从一台服务器到两台 | `114` → `122` | 仍是 `-np 4`，用 rank 映射确认位置 |
| 改数据规模 | 测试用例参数 `-b`、`-e` | 参与者不变，通信域和 `-np` 不动 |
| 换算子 | 换成对应的测试用例程序 | 确认该测试用例存在，核对算子所需参数 |

In [ ]:
%%bash
source ~/hccl_vm_pack/env.sh

tmux send-keys -t hvm "mpirun --oversubscribe -np 4 $TEST_BIN/reduce_scatter_test -b 64 -e 64 -d int32 -o sum -w 0 -n 1 -c 1 > $VM_INSTALL/bin/log_122.txt 2>&1" Enter
for i in $(seq 1 30); do
    grep -q "check_result" "$VM_INSTALL/bin/log_122.txt" 2>/dev/null && break
    sleep 2
done

echo "--- result table ---"
grep -n -A 3 "check_result" "$VM_INSTALL/bin/log_122.txt"
echo "--- error count ---"
grep -cE "\[(error|fatal)\]" "$VM_INSTALL/bin/log_122.txt" || true

`check_result` 显示 `failed` 属正常（模拟器不算数，见 03.04 第 5 节）。
末尾的 `FreePhyMem` 是进程退出时释放模拟物理内存的正常日志，不是报错。

接着触发 Checker：

In [ ]:
%%bash
source ~/hccl_vm_pack/env.sh

tmux send-keys -t hvm "$VM plugin run @checker" Enter
for i in $(seq 1 30); do
    tmux capture-pane -pt hvm | grep -q "CHECKER_RUN_SUMMARY" && break
    sleep 2
done
tmux capture-pane -pt hvm | grep -E "Checker result|CHECKER_RUN_SUMMARY|Total Op"

期望 `Total Op: 4`（四个 rank 各一条），并且 `All Success`。

跑完把会话关掉，避免它一直占着资源：

In [ ]:
%%bash
tmux kill-server 2>/dev/null
echo "session cleaned"

## 7. 遇到问题先看这里

| 现象 | 原因 | 处理 |
| :--- | :--- | :--- |
| `mock-comm` 报 `copy topo.json failed` | 不在 start 的会话里执行，`g_configClusterDir` 为空 | 必须用 `tmux send-keys` 送进会话，不要在普通终端直接跑 |
| `data/ranktable.json` 不存在 | `mock-comm` 没成功 | 重跑第 3 节最后那个单元 |
| `Total Op: 0` | 测试用例没执行到 | 看 `log_122.txt` 里有没有 `Entry-HcclCommInitClusterInfo` |
| `-np 4` 但 Checker 只有 2 条 | 通信域还是 112 | 先 `mock-comm 122` 再跑测试用例 |
| 单元一直不返回 | 直接执行了 `start` | `tmux kill-server` 清掉，按 2.3 用 tmux 起 |

[← 3.5 跑通第一个通信测试](./03.05_first_comm_test.ipynb) · [返回本章目录](./03.01_chapter_intro.ipynb) · [3.7 通信用例实战 →](./03.07_communication_cases.ipynb)